# Training the RNNs (PRE and POST)

Trains the recurrent networks used in the RNN analyses of the paper, on the PRE and POST conditions.
Each network is seeded with its own child of `SeedSequence(seed_base)`, so training is reproducible.
Run in the `NEURCOMP` virtual environment (`../environment.yml`).

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import platform
import numpy as np
import torch
from joblib import Parallel, delayed

sys.path.insert(0, os.path.abspath('..'))  # func/ is in python/
from func import *

print(platform.platform())
print(sys.executable)  # should be the NEURCOMP environment
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
# training configuration (same for all networks)
cfg = {
    'input_size':  5,                  # reference cos/sin, stimulus cos/sin, GO signal
    'hidden_size': 32,                 # number of hidden units
    'output_size': 1,                  # one logit: >= 0 for category 1
    'batch_size':  512,                # sequences per sequence length at each update (half PRE, half POST)
    'nbatch':      1,                  # gradient updates per epoch
    'max_epochs':  1000,               # number of epochs
    'alpha':       0.001,              # learning rate (Adam)
    'savecheck':   20,                 # network saved every 20 epochs (epo_0000, epo_0020, ..., epo_1000)
    'saveornot':   True,               # save the trained networks
    'patience':    0,                  # no early stopping
    'device':      device,
    'condn_train': np.array([0, 1]),   # PRE (0) and POST (1)
    'seqlen':      np.arange(2, 13),   # numbers of stimuli per sequence: 2 to 12
    'noise_value': 0,                  # no recurrent noise during training
    'sigcat':      15,                 # s.d. of the reference noise in PRE (deg)
    'seed_base':   0,                 # master seed: network irnn is seeded with the irnn-th child of SeedSequence(seed_base)
}
nrnn = 100  # number of networks

# output folder (networks trained with noise: run_training saves them in sigcat_X_noise_X-XX)
cfg['fname_train'] = os.path.join('..', 'rnn_simulations', 'RNN_train_PRE_POST', f'sigcat_{cfg["sigcat"]}')
print(cfg['fname_train'])

In [ ]:
# train the networks in parallel (network irnn is seeded with the irnn-th child of SeedSequence(seed_base))
# each network is saved in cfg['fname_train'] (cfg['fname_train'] + '_noise_X-XX' if trained with noise) as model_irnn_XX.tar
# and saved every savecheck epochs as model_irnn_XX_epo_0000.pt (untrained), ..., _epo_1000.pt
results = Parallel(n_jobs=-1)(
    delayed(run_training)(irnn, cfg['noise_value'], cfg, verbose=False) for irnn in range(nrnn))

# accuracy at the last epoch (%, last batch) w.r.t. the generative category and the optimal choice
acc    = np.array([res[0]['acc'][-1] for res in results])
accopt = np.array([res[0]['accopt'][-1] for res in results])
print(f'accuracy (generative category): {acc.mean():.1f} ± {acc.std():.1f} %')
print(f'accuracy (optimal choice):      {accopt.mean():.1f} ± {accopt.std():.1f} %')